# Amazon ML Challenge: Entity Resolution
This notebook contains the complete End-to-End Pipeline.
It reconstructs the local `src/` modular architecture directly on the Kaggle environment.

In [ ]:
!mkdir -p src/preprocessing src/blocking src/evaluation src/features src/matching src/postprocessing
!touch src/__init__.py src/blocking/__init__.py src/evaluation/__init__.py src/features/__init__.py src/matching/__init__.py src/postprocessing/__init__.py
!pip install unidecode scikit-learn pandas scipy thefuzz jellyfish lightgbm sparse-dot-topn

In [ ]:
%%writefile src/config.py
import os
from pathlib import Path

# Base Paths
BASE_DIR = Path(__file__).resolve().parent.parent
DATA_DIR = BASE_DIR / "6ab10eb3b23ba_student_resource" / "student_resource" / "dataset"
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
OUTPUT_DIR = BASE_DIR / "output"

# File Paths
TRAIN_S1 = TRAIN_DIR / "train_source1.tsv"
TRAIN_S2 = TRAIN_DIR / "train_source2.tsv"
TRAIN_S3 = TRAIN_DIR / "train_source3.tsv"
TRAIN_GT = TRAIN_DIR / "train_ground_truth.tsv"

TEST_S1 = TEST_DIR / "test_source1.tsv"
TEST_S2 = TEST_DIR / "test_source2.tsv"
TEST_S3 = TEST_DIR / "test_source3.tsv"

# Pipeline Parameters
NUM_WORKERS = os.cpu_count() or 4
CHUNK_SIZE = 100_000

# Blocking Parameters
TFIDF_TOP_K = 50
EMBEDDING_TOP_K = 20

# Model Parameters
SINGLETON_THRESHOLD = 0.5  # To be tuned
MATCH_THRESHOLD = 0.5      # To be tuned



In [ ]:
%%writefile src/preprocessing/__init__.py
from .transliteration import transliterate_text, transliterate_series
from .text_normalizer import normalize_text, normalize_series
from .name_normalizer import normalize_name, normalize_name_series
from .country_normalizer import normalize_country, normalize_country_series
from .address_parser import apply_address_parsing, parse_address_regex

__all__ = [
    'transliterate_text',
    'transliterate_series',
    'normalize_text',
    'normalize_series',
    'normalize_name',
    'normalize_name_series',
    'normalize_country',
    'normalize_country_series',
    'apply_address_parsing',
    'parse_address_regex'
]



In [ ]:
%%writefile src/preprocessing/transliteration.py
from unidecode import unidecode
import pandas as pd

# Empirical mapping of Devanagari loanwords to standard English spellings
PHONETIC_MAPPING = {
    'limittedd': 'limited',
    'limittett': 'limited',
    'limirrrrdd': 'limited',
    'praaivett': 'private',
    'praiveett': 'private',
    'praaibhett': 'private',
    'piraiveett': 'private',
    'praivrrrr': 'private',
    'praa': 'private',
    'li': 'limited',
    'elelpii': 'llp',
    'ttek': 'tech',
    'inttrneshnl': 'international',
    'globl': 'global',
    'inphraa': 'infra',
    'proddktts': 'products',
    'bilddrs': 'builders',
    'inphraasttrkcr': 'infrastructure',
    'knsltting': 'consulting',
    'ddevlprs': 'developers',
    'investtmentt': 'investment',
    'sonlyuushns': 'solutions',
    'knsttrkshns': 'constructions',
    'phuudds': 'foods',
    'phuudd': 'food',
    'injiiniyring': 'engineering',
    'ttredding': 'trading',
    'inddsttriij': 'industries',
    'enttrpraaijej': 'enterprises',
    'tteknolonjiij': 'technologies',
    'tteknolonjii': 'technology',
    'lkssmii': 'lakshmi',
    'shrii': 'shri',
    'vencrs': 'ventures',
    'esttett': 'estate',
    'sonphttveyr': 'software',
    'eksportts': 'exports',
    'bijnes': 'business',
    'phaaunddeshn': 'foundation',
    'sisttms': 'systems',
    'inphottek': 'infotech',
    'impeks': 'impex',
    'phaainens': 'finance',
    'aaiittii': 'it',
    'mainejmentt': 'management',
    'maarketting': 'marketing',
    'enrjii': 'energy',
    'proddyuusr': 'producer',
    'projektts': 'projects',
    'srvisej': 'services',
    'egro': 'agro',
    'pronprttiij': 'properties',
    'knslttensii': 'consultancy',
    'paavr': 'power',
    'honspittailittii': 'hospitality',
    'aaiinnsii': 'inc',
    'kaNpnii': 'company',
    'kampnii': 'company',
    'kompnii': 'company',
    'kaarporeshn': 'corporation',
}

def transliterate_text(text: str) -> str:
    """
    Transliterates non-Latin scripts (Devanagari, Kannada, etc.) to Latin characters.
    Uses unidecode which handles almost all Unicode characters.
    Applies phonetic mapping to correct common English loanwords.
    """
    if pd.isna(text):
        return ""
    
    # Fast path: check if it only has ascii
    try:
        text.encode('ascii')
        return text
    except UnicodeEncodeError:
        trans = unidecode(str(text))
        # Apply word-by-word mapping for loanwords
        words = trans.split()
        mapped = [PHONETIC_MAPPING.get(w.lower(), w) for w in words]
        return " ".join(mapped)

def transliterate_series(series: pd.Series) -> pd.Series:
    """Applies transliteration to a pandas Series efficiently."""
    return series.apply(transliterate_text)



In [ ]:
%%writefile src/preprocessing/text_normalizer.py
import re
import pandas as pd
from .transliteration import transliterate_text

def normalize_text(text: str) -> str:
    """
    Basic text normalization:
    1. Lowercase
    2. Transliterate to Latin
    3. Remove punctuation and extra spaces
    """
    if pd.isna(text) or str(text).strip() == "":
        return ""
    
    text = transliterate_text(str(text))
    text = text.lower()
    
    # Keep only alphanumeric and spaces
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_series(series: pd.Series) -> pd.Series:
    return series.apply(normalize_text)



In [ ]:
%%writefile src/preprocessing/country_normalizer.py
import pandas as pd

def normalize_country(country: str) -> str:
    """
    Standardize the country field.
    The problem statement specifies it must remain an open-set string filter.
    France is explicitly mentioned as being in the test set.
    We just clean whitespace and lowercase it for reliable matching.
    """
    if pd.isna(country) or str(country).strip() == "":
        return "unknown"
    return str(country).strip().lower()

def normalize_country_series(series: pd.Series) -> pd.Series:
    return series.apply(normalize_country)



In [ ]:
%%writefile src/preprocessing/name_normalizer.py
import re
import pandas as pd

# Common business abbreviations mapping
ABBREVIATIONS = {
    r'\bltd\b': 'limited',
    r'\bpvt\b': 'private',
    r'\binc\b': 'incorporated',
    r'\bcorp\b': 'corporation',
    r'\bco\b': 'company',
    r'\bllc\b': 'limited liability company',
    r'\bllp\b': 'limited liability partnership',
    r'\b&\b': 'and',
}

def expand_abbreviations(text: str) -> str:
    """Expands common business abbreviations for standardisation."""
    if not text:
        return ""
    
    # Assumes text is already lowercased and transliterated
    for abbr, expanded in ABBREVIATIONS.items():
        text = re.sub(abbr, expanded, text)
        
    # Standardize 'dba' (doing business as)
    text = re.sub(r'\bdba\b', 'doing business as', text)
    
    # Remove website domain suffixes like .com, .in (from our EDA, we saw www.shivshakti.com)
    text = re.sub(r'\bwww\.\S+', '', text)
    text = re.sub(r'\S+\.com\b', '', text)
    
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_name(text: str) -> str:
    """Pipeline for name normalization"""
    if pd.isna(text) or str(text).strip() == "":
        return ""
    
    # Expand abbreviations
    text = expand_abbreviations(str(text))
    return text

def normalize_name_series(series: pd.Series) -> pd.Series:
    return series.apply(normalize_name)



In [ ]:
%%writefile src/preprocessing/address_parser.py
import re
import pandas as pd

def extract_pincode(address: str) -> str:
    """Extracts PIN/ZIP code from address string"""
    if not address:
        return ""
    # Look for 5-digit (US) or 6-digit (India) numbers, often at the end
    match = re.search(r'\b(\d{5,6})\b', str(address))
    if match:
        return match.group(1)
    return ""

def extract_numbers(address: str) -> str:
    """Extracts all numbers (house numbers, street numbers) from address"""
    if not address:
        return ""
    numbers = re.findall(r'\b\d+\b', str(address))
    return " ".join(numbers)

def extract_street_type(address: str) -> str:
    """Extracts common street types to match across abbreviations"""
    if not address:
        return ""
    types = []
    if re.search(r'\b(st|street)\b', address): types.append('street')
    if re.search(r'\b(rd|road)\b', address): types.append('road')
    if re.search(r'\b(ave|avenue)\b', address): types.append('avenue')
    if re.search(r'\b(blvd|boulevard)\b', address): types.append('boulevard')
    if re.search(r'\b(dr|drive)\b', address): types.append('drive')
    if re.search(r'\b(ln|lane)\b', address): types.append('lane')
    if re.search(r'\b(ct|court)\b', address): types.append('court')
    if re.search(r'\b(pl|place)\b', address): types.append('place')
    if re.search(r'\b(sq|square)\b', address): types.append('square')
    if re.search(r'\b(hwy|highway)\b', address): types.append('highway')
    return " ".join(types)

def extract_state_code(address: str) -> str:
    """Extracts common state codes (very naive, can be expanded)"""
    if not address:
        return ""
    # Only 2 letter uppercase words bounded by spaces/punctuation (done before lowercasing)
    # Actually, we assume input is already normalized (lowercased)
    # We will skip this for now or rely on TF-IDF word matches
    return ""

def parse_address_regex(address: str) -> dict:
    """
    Regex fallback for parsing addresses since libpostal might be too slow.
    Assumes address is already normalized (lowercased, transliterated, punctuation removed).
    """
    if pd.isna(address) or str(address).strip() == "":
        return {
            'pincode': '',
            'numbers': '',
            'street_type': ''
        }
    
    return {
        'pincode': extract_pincode(address),
        'numbers': extract_numbers(address),
        'street_type': extract_street_type(address)
    }

def apply_address_parsing(df: pd.DataFrame, address_col: str = 'norm_address') -> pd.DataFrame:
    """Applies regex address parsing and adds extracted components as new columns"""
    parsed = df[address_col].apply(parse_address_regex)
    df_parsed = pd.DataFrame(parsed.tolist(), index=df.index)
    
    # Prefix columns
    df_parsed = df_parsed.add_prefix('addr_')
    return pd.concat([df, df_parsed], axis=1)



In [ ]:
%%writefile src/data_loader.py
import pandas as pd
import numpy as np
import time
from . import config
from .preprocessing import (
    normalize_series,
    normalize_name_series,
    normalize_country_series,
    apply_address_parsing
)

def load_and_preprocess(filepath, is_ground_truth=False):
    """
    Loads a TSV file and applies the full preprocessing pipeline.
    """
    print(f"Loading {filepath}...")
    t0 = time.time()
    df = pd.read_csv(filepath, sep='\t', na_filter=False, dtype=str)
    print(f"  Loaded {len(df):,} rows in {time.time()-t0:.1f}s")
    
    if is_ground_truth:
        return df
    
    # 1. Normalize Country
    print("  Normalizing country...")
    t0 = time.time()
    df['country'] = normalize_country_series(df['country'])
    
    # 2. Normalize and Transliterate Name
    print("  Normalizing name (transliteration + lowercasing)...")
    t0 = time.time()
    df['norm_name'] = normalize_series(df['business_name'])
    df['norm_name'] = normalize_name_series(df['norm_name'])
    
    # 3. Normalize and Transliterate Address
    print("  Normalizing address (transliteration + lowercasing)...")
    t0 = time.time()
    df['norm_address'] = normalize_series(df['business_address'])
    
    # 4. Extract Address Components (Regex)
    print("  Extracting address components...")
    t0 = time.time()
    df = apply_address_parsing(df, address_col='norm_address')
    
    print(f"  Preprocessing complete. Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    
    return df

def test_pipeline():
    """Quick test on a tiny sample"""
    sample_data = {
        'entity_id': ['S1-1', 'S2-2', 'S3-3'],
        'business_name': ['Dahlia Power Reliable Scientific LLC', 'राम मार्केटिंग प्राइवेट लिमिटेड', 'www.shivshakti.com'],
        'business_address': ['630 45th Terrace, Kansas City, MO', 'KH NO. -570/13, NEW DELHI', ''],
        'country': ['US', 'India', 'India']
    }
    df = pd.DataFrame(sample_data)
    
    df['country'] = normalize_country_series(df['country'])
    df['norm_name'] = normalize_series(df['business_name'])
    df['norm_name'] = normalize_name_series(df['norm_name'])
    df['norm_address'] = normalize_series(df['business_address'])
    df = apply_address_parsing(df, address_col='norm_address')
    
    print("Pipeline Test Results:")
    for _, row in df.iterrows():
        print(f"\nOriginal: {row['business_name']} | {row['business_address']}")
        print(f"Name:     {row['norm_name']}")
        print(f"Address:  {row['norm_address']}")
        print(f"Parsed:   PIN={row['addr_pincode']} NUMS={row['addr_numbers']} TYPE={row['addr_street_type']}")

if __name__ == "__main__":
    test_pipeline()



In [ ]:
%%writefile src/blocking/country_partition.py
import pandas as pd

def get_country_partitions(df: pd.DataFrame, country_col: str = 'country') -> dict:
    """
    Groups the dataframe by country to create independent blocking partitions.
    Returns a dictionary mapping country names to boolean masks or dataframes.
    For extreme scale, returning indices or masks is more memory efficient.
    """
    partitions = {}
    for country in df[country_col].unique():
        # Keep track of indices where country matches
        partitions[country] = df[country_col] == country
        
    return partitions

def filter_by_country(df: pd.DataFrame, country: str, country_col: str = 'country') -> pd.DataFrame:
    """Returns subset of df for a specific country."""
    return df[df[country_col] == country].copy()



In [ ]:
%%writefile src/blocking/indexer.py
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np
import scipy.sparse as sp

class TFIDFIndexer:
    def __init__(self, analyzer='char_wb', ngram_range=(2, 4), min_df=1, max_df=0.9):
        """
        Initializes the TF-IDF Vectorizer.
        We use character n-grams (default 2-4) because they are highly robust to
        spelling variations, typos, and fuzzy matching compared to word n-grams.
        """
        self.vectorizer = TfidfVectorizer(
            analyzer=analyzer,
            ngram_range=ngram_range,
            min_df=min_df,
            max_df=max_df,
            lowercase=False, # Input is already lowercased from preprocessing
            dtype=np.float32 # Save massive amount of RAM compared to default float64
        )
        self.is_fitted = False

    def build_corpus(self, df: pd.DataFrame, text_col: str) -> pd.Series:
        """Fills NaNs to prevent vectorizer errors."""
        return df[text_col].fillna("")

    def fit(self, df: pd.DataFrame, text_col: str = 'norm_name'):
        """Fits the TF-IDF vocabulary on the dataset."""
        corpus = self.build_corpus(df, text_col)
        self.vectorizer.fit(corpus)
        self.is_fitted = True

    def transform(self, df: pd.DataFrame, text_col: str = 'norm_name') -> sp.csr_matrix:
        """Transforms the dataset into sparse TF-IDF vectors."""
        if not self.is_fitted:
            raise ValueError("Indexer must be fitted before transform.")
        
        corpus = self.build_corpus(df, text_col)
        return self.vectorizer.transform(corpus)
        
    def fit_transform(self, df: pd.DataFrame, text_col: str = 'norm_name') -> sp.csr_matrix:
        """Fits and transforms the dataset."""
        corpus = self.build_corpus(df, text_col)
        self.is_fitted = True
        return self.vectorizer.fit_transform(corpus)



In [ ]:
%%writefile src/blocking/candidate_generator.py
import numpy as np
import scipy.sparse as sp
import pandas as pd
import time
from sparse_dot_topn import sp_matmul_topn

class CandidateGenerator:
    def __init__(self, top_k=50):
        self.top_k = top_k
        
    def generate_candidates(self, df_s1: pd.DataFrame, df_s2: pd.DataFrame, 
                            s1_matrix: sp.csr_matrix, s2_matrix: sp.csr_matrix, 
                            s2_prefix: str = 'S2') -> pd.DataFrame:
        """
        Generates pairs of candidates from S1 and S2 using sparse_dot_topn.
        This avoids dense matrix explosion by computing only the top K similarities in C++.
        """
        print(f"  Generating candidates for {len(df_s1)} queries against {len(df_s2)} targets...")
        t0 = time.time()
        
        # Ensure matrices are float32 to save memory
        s1_matrix = s1_matrix.astype(np.float32)
        s2_matrix = s2_matrix.astype(np.float32)
        
        # We need S2 transposed. sp_matmul_topn works well with B as CSR.
        s2_matrix_T = s2_matrix.T.tocsr()
        
        # Compute Top K
        # threshold=0.0 means we keep anything > 0.0 (no overlap = discarded)
        print(f"  Running sparse_dot_topn for top {self.top_k}...")
        C = sp_matmul_topn(s1_matrix, s2_matrix_T, top_n=self.top_k, threshold=0.0001, n_threads=4)
        
        print(f"  Computed sparse dot products in {time.time()-t0:.1f}s")
        
        # C is a CSR matrix of shape (num_queries, num_db)
        # We extract the non-zero indices. 
        # C.nonzero() returns (row_indices, col_indices)
        row_indices, col_indices = C.nonzero()
        scores = C.data
        
        # Map indices back to entity IDs
        s1_ids = df_s1['entity_id'].values
        s2_ids = df_s2['entity_id'].values
        
        candidates_df = pd.DataFrame({
            's1_entity_id': s1_ids[row_indices],
            f'{s2_prefix}_entity_id': s2_ids[col_indices],
            'blocking_score': scores
        })
        
        return candidates_df



In [ ]:
%%writefile src/blocking/pipeline.py
import pandas as pd
from typing import Dict, List
import time

from .country_partition import get_country_partitions
from .indexer import TFIDFIndexer
from .candidate_generator import CandidateGenerator

class BlockingPipeline:
    def __init__(self, top_k=50, ngram_range=(2, 4)):
        self.top_k = top_k
        self.ngram_range = ngram_range
        
    def run(self, df_s1: pd.DataFrame, df_s2: pd.DataFrame, s2_prefix: str = 's2') -> pd.DataFrame:
        """
        Runs the full blocking pipeline:
        1. Partitions data by country.
        2. Fits TF-IDF on S2 partition.
        3. Transforms S1 and S2 partitions.
        4. Generates top K candidates per query in S1.
        Returns a single unified DataFrame of all candidates.
        """
        print(f"\n--- Starting Blocking Pipeline vs {s2_prefix.upper()} ---")
        t0 = time.time()
        
        # Partition by country
        s1_partitions = get_country_partitions(df_s1)
        s2_partitions = get_country_partitions(df_s2)
        
        all_candidates = []
        
        # We only care about countries present in S1
        for country in s1_partitions.keys():
            df_s1_c = df_s1[s1_partitions[country]]
            
            # If the country doesn't exist in S2, no candidates can be found
            if country not in s2_partitions:
                print(f"Skipping {country}: No records in {s2_prefix.upper()}.")
                continue
                
            df_s2_c = df_s2[s2_partitions[country]]
            print(f"Processing partition: {country} (S1: {len(df_s1_c)}, {s2_prefix.upper()}: {len(df_s2_c)})")
            
            # TF-IDF
            # Note: We fit TF-IDF only on S2 partition to build the vocabulary of the target database
            indexer = TFIDFIndexer(ngram_range=self.ngram_range, min_df=1) # min_df=1 for small partitions
            
            s2_matrix = indexer.fit_transform(df_s2_c)
            s1_matrix = indexer.transform(df_s1_c)
            
            # Candidate Generation
            generator = CandidateGenerator(top_k=self.top_k)
            candidates = generator.generate_candidates(
                df_s1_c, df_s2_c, s1_matrix, s2_matrix, s2_prefix=s2_prefix
            )
            
            all_candidates.append(candidates)
            
        if not all_candidates:
            return pd.DataFrame()
            
        final_candidates = pd.concat(all_candidates, ignore_index=True)
        print(f"--- Blocking Pipeline Complete: Generated {len(final_candidates)} candidates in {time.time()-t0:.1f}s ---")
        return final_candidates



In [ ]:
%%writefile src/features/name_features.py
import pandas as pd
from thefuzz import fuzz
import jellyfish

def compute_name_features(df: pd.DataFrame, prefix1='s1', prefix2='S2') -> pd.DataFrame:
    """
    Computes name similarity features between S1 and S2 candidates.
    df must contain columns: <prefix1>_norm_name, <prefix2>_norm_name
    """
    col1 = f"{prefix1}_norm_name"
    col2 = f"{prefix2}_norm_name"
    
    # We use a vectorized approach or list comprehension for speed
    names1 = df[col1].fillna("").astype(str).tolist()
    names2 = df[col2].fillna("").astype(str).tolist()
    
    # Initialize feature lists
    f_jw = []
    f_lev = []
    f_token_sort = []
    f_token_set = []
    f_len_diff = []
    
    for n1, n2 in zip(names1, names2):
        if not n1 and not n2:
            f_jw.append(-1.0)
            f_lev.append(-1.0)
            f_token_sort.append(-1)
            f_token_set.append(-1)
            f_len_diff.append(-1)
            continue
        if not n1 or not n2:
            f_jw.append(0.0)
            f_lev.append(0.0)
            f_token_sort.append(0)
            f_token_set.append(0)
            f_len_diff.append(abs(len(n1) - len(n2)))
            continue
            
        f_jw.append(jellyfish.jaro_winkler_similarity(n1, n2))
        
        # Levenshtein distance normalized by max length
        max_len = max(len(n1), len(n2))
        dist = jellyfish.levenshtein_distance(n1, n2)
        f_lev.append(1.0 - (dist / max_len) if max_len > 0 else 0)
        
        f_token_sort.append(fuzz.token_sort_ratio(n1, n2))
        f_token_set.append(fuzz.token_set_ratio(n1, n2))
        
        f_len_diff.append(abs(len(n1) - len(n2)))
        
    features = pd.DataFrame({
        'name_jaro_winkler': f_jw,
        'name_norm_levenshtein': f_lev,
        'name_token_sort_ratio': f_token_sort,
        'name_token_set_ratio': f_token_set,
        'name_len_diff': f_len_diff
    })
    
    return features



In [ ]:
%%writefile src/features/address_features.py
import pandas as pd
from thefuzz import fuzz
import jellyfish

def compute_address_features(df: pd.DataFrame, prefix1='s1', prefix2='S2') -> pd.DataFrame:
    """
    Computes similarity features between S1 and S2 address components.
    df must contain columns:
        <prefix1>_norm_address, <prefix2>_norm_address
        <prefix1>_addr_pincode, <prefix2>_addr_pincode
        <prefix1>_addr_numbers, <prefix2>_addr_numbers
        <prefix1>_addr_street_type, <prefix2>_addr_street_type
    """
    features = pd.DataFrame()
    
    # Full Address Similarity
    add1 = df[f"{prefix1}_norm_address"].fillna("").astype(str).tolist()
    add2 = df[f"{prefix2}_norm_address"].fillna("").astype(str).tolist()
    
    f_addr_jw = []
    f_addr_tokenset = []
    
    for a1, a2 in zip(add1, add2):
        if not a1 and not a2:
            f_addr_jw.append(-1.0)
            f_addr_tokenset.append(-1)
            continue
        if not a1 or not a2:
            f_addr_jw.append(0.0)
            f_addr_tokenset.append(0)
            continue
            
        f_addr_jw.append(jellyfish.jaro_winkler_similarity(a1, a2))
        f_addr_tokenset.append(fuzz.token_set_ratio(a1, a2))
        
    features['addr_jaro_winkler'] = f_addr_jw
    features['addr_token_set_ratio'] = f_addr_tokenset
    
    # Exact Match Features (Pincode, Numbers, Street Type)
    for comp in ['pincode', 'numbers', 'street_type']:
        c1 = df[f"{prefix1}_addr_{comp}"].fillna("").astype(str)
        c2 = df[f"{prefix2}_addr_{comp}"].fillna("").astype(str)
        
        # 1 if exact match, 0 if mismatch, -1 if either is missing
        exact_match = (c1 == c2).astype(int)
        missing_mask = (c1 == "") | (c2 == "")
        exact_match[missing_mask] = -1
        
        features[f'{comp}_exact_match'] = exact_match.values
        
    return features



In [ ]:
%%writefile src/features/phonetic_features.py
import pandas as pd
import jellyfish
from thefuzz import fuzz

def compute_phonetic_features(df: pd.DataFrame, prefix1='s1', prefix2='S2') -> pd.DataFrame:
    """
    Computes phonetic similarity features (Soundex and Match Rating Approach).
    Useful for catching names that sound identical but are spelled very differently.
    """
    col1 = f"{prefix1}_norm_name"
    col2 = f"{prefix2}_norm_name"
    
    names1 = df[col1].fillna("").astype(str).tolist()
    names2 = df[col2].fillna("").astype(str).tolist()
    
    f_soundex_match = []
    f_mra_match = []
    
    for n1, n2 in zip(names1, names2):
        if not n1 and not n2:
            f_soundex_match.append(1)
            f_mra_match.append(1)
            continue
            
        # Get first word of the name for phonetic comparison (usually the defining noun)
        w1 = n1.split()[0] if n1 else ""
        w2 = n2.split()[0] if n2 else ""
        
        # Jellyfish strictly requires alphabetical characters for phonetic algorithms
        alpha_w1 = ''.join(c for c in w1 if c.isalpha()) if w1 else ""
        alpha_w2 = ''.join(c for c in w2 if c.isalpha()) if w2 else ""
        
        # Soundex
        s1 = ""
        s2 = ""
        try:
            s1 = jellyfish.soundex(alpha_w1) if alpha_w1 else ""
            s2 = jellyfish.soundex(alpha_w2) if alpha_w2 else ""
        except ValueError:
            pass
            
        f_soundex_match.append(1 if (s1 and s1 == s2) else 0)
        
        # Match Rating Approach (MRA)
        m1 = ""
        m2 = ""
        try:
            m1 = jellyfish.match_rating_codex(alpha_w1) if alpha_w1 else ""
            m2 = jellyfish.match_rating_codex(alpha_w2) if alpha_w2 else ""
        except ValueError:
            pass
        
        # Fuzzy match on the MRA code
        f_mra_match.append(fuzz.ratio(m1, m2))
        
    features = pd.DataFrame({
        'phonetic_soundex_match': f_soundex_match,
        'phonetic_mra_ratio': f_mra_match
    })
    
    return features



In [ ]:
%%writefile src/features/cross_features.py
import pandas as pd
import jellyfish
from thefuzz import fuzz

def compute_cross_features(df: pd.DataFrame, prefix1='s1', prefix2='S2') -> pd.DataFrame:
    """
    Computes cross-field features.
    Sometimes a Name is accidentally placed in the Address field (or vice versa).
    We check the similarity of S1 Name vs S2 Address, and S1 Address vs S2 Name.
    """
    name1 = df[f"{prefix1}_norm_name"].fillna("").astype(str).tolist()
    addr1 = df[f"{prefix1}_norm_address"].fillna("").astype(str).tolist()
    
    name2 = df[f"{prefix2}_norm_name"].fillna("").astype(str).tolist()
    addr2 = df[f"{prefix2}_norm_address"].fillna("").astype(str).tolist()
    
    f_n1_a2 = []
    f_a1_n2 = []
    
    for n1, a1, n2, a2 in zip(name1, addr1, name2, addr2):
        # S1 Name vs S2 Address
        if n1 and a2:
            f_n1_a2.append(fuzz.token_set_ratio(n1, a2))
        else:
            f_n1_a2.append(0)
            
        # S1 Address vs S2 Name
        if a1 and n2:
            f_a1_n2.append(fuzz.token_set_ratio(a1, n2))
        else:
            f_a1_n2.append(0)
            
    features = pd.DataFrame({
        'cross_name1_addr2_tokenset': f_n1_a2,
        'cross_addr1_name2_tokenset': f_a1_n2
    })
    
    return features



In [ ]:
%%writefile src/features/feature_pipeline.py
import pandas as pd
from .name_features import compute_name_features
from .address_features import compute_address_features
from .phonetic_features import compute_phonetic_features
from .cross_features import compute_cross_features

def build_features(df_pairs: pd.DataFrame, df_s1: pd.DataFrame, df_s2: pd.DataFrame, s2_prefix: str = 'S2') -> pd.DataFrame:
    """
    Given a dataframe of candidate pairs (s1_entity_id, S2_entity_id, blocking_score),
    joins the original text data and computes all ML features.
    """
    print("Building features for candidate pairs...")
    
    # Only keep the necessary columns to save RAM (drop raw text)
    cols_to_keep = ['entity_id', 'norm_name', 'norm_address', 'addr_pincode', 'addr_numbers', 'addr_street_type']
    
    df_s1_slim = df_s1[[c for c in cols_to_keep if c in df_s1.columns]]
    df_s2_slim = df_s2[[c for c in cols_to_keep if c in df_s2.columns]]
    
    # 1. Join S1 data
    df_s1_renamed = df_s1_slim.add_prefix('s1_')
    merged = df_pairs.merge(df_s1_renamed, on='s1_entity_id', how='left')
    
    # 2. Join S2 data
    df_s2_renamed = df_s2_slim.add_prefix(f'{s2_prefix}_')
    merged = merged.merge(df_s2_renamed, on=f'{s2_prefix}_entity_id', how='left')
    
    # 3. Compute Features
    print("  Computing name features...")
    name_feats = compute_name_features(merged, prefix1='s1', prefix2=s2_prefix)
    
    print("  Computing address features...")
    addr_feats = compute_address_features(merged, prefix1='s1', prefix2=s2_prefix)
    
    print("  Computing phonetic features...")
    phonetic_feats = compute_phonetic_features(merged, prefix1='s1', prefix2=s2_prefix)
    
    print("  Computing cross-field features...")
    cross_feats = compute_cross_features(merged, prefix1='s1', prefix2=s2_prefix)
    
    # 4. Combine all features
    # Start with the blocking score as the first feature
    feature_matrix = pd.DataFrame({'blocking_score': merged['blocking_score']})
    feature_matrix = pd.concat([
        feature_matrix, 
        name_feats, 
        addr_feats,
        phonetic_feats,
        cross_feats
    ], axis=1)
    
    return feature_matrix



In [ ]:
%%writefile src/matching/lgbm_matcher.py
import lightgbm as lgb
import pandas as pd
import numpy as np

class LGBMMatcher:
    def __init__(self, params=None):
        """
        Initializes the LightGBM matching model.
        Default parameters are tuned for binary classification with unbalanced data.
        """
        if params is None:
            self.params = {
                'objective': 'binary',
                'metric': 'binary_logloss',
                'boosting_type': 'gbdt',
                'learning_rate': 0.05,
                'num_leaves': 31,
                'max_depth': -1,
                'feature_fraction': 0.8,
                'n_estimators': 300,
                'random_state': 42,
                'n_jobs': -1
            }
        else:
            self.params = params
            
        self.model = lgb.LGBMClassifier(**self.params)
        self.is_fitted = False

    def fit(self, X_train: pd.DataFrame, y_train: pd.Series, X_val=None, y_val=None):
        """
        Trains the LightGBM model. 
        If validation data is provided, it uses early stopping.
        """
        print("Training LightGBM Matcher...")
        eval_set = [(X_train, y_train)]
        if X_val is not None and y_val is not None:
            eval_set.append((X_val, y_val))
            
        self.model.fit(
            X_train, 
            y_train,
            eval_set=eval_set,
            eval_metric='binary_logloss'
        )
        self.is_fitted = True
        print("Training complete.")

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        """Returns the probability that the pair is a MATCH."""
        if not self.is_fitted:
            raise ValueError("Model is not fitted yet.")
        
        # predict_proba returns [prob_class_0, prob_class_1]
        return self.model.predict_proba(X)[:, 1]
    
    def get_feature_importances(self) -> pd.DataFrame:
        """Returns a dataframe of feature importances."""
        if not self.is_fitted:
            return pd.DataFrame()
            
        return pd.DataFrame({
            'feature': self.model.feature_name_,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)



In [ ]:
%%writefile src/matching/singleton_detector.py
import pandas as pd

def apply_singleton_threshold(predictions_df: pd.DataFrame, singleton_threshold=0.6, match_threshold=0.4) -> pd.DataFrame:
    """
    Applies the singleton-aware decision logic.
    For each query in S1:
    1. If the BEST match probability is < singleton_threshold, we return EMPTY (singleton).
    2. Otherwise, we return ALL matches with probability >= match_threshold.
    
    predictions_df must contain: ['s1_entity_id', 'S2_entity_id', 'match_prob']
    Returns a dataframe with the final selected matches.
    """
    # 1. Find the best match probability for each S1 entity
    best_probs = predictions_df.groupby('s1_entity_id')['match_prob'].max().reset_index()
    best_probs.rename(columns={'match_prob': 'best_prob'}, inplace=True)
    
    # 2. Join the best probability back to the predictions
    merged = predictions_df.merge(best_probs, on='s1_entity_id', how='left')
    
    # 3. Apply logic
    # Keep only rows where best_prob >= singleton_threshold AND match_prob >= match_threshold
    mask = (merged['best_prob'] >= singleton_threshold) & (merged['match_prob'] >= match_threshold)
    
    final_matches = merged[mask].copy()
    
    # Drop the temporary column
    final_matches.drop(columns=['best_prob'], inplace=True)
    
    return final_matches



In [ ]:
%%writefile src/postprocessing/submission_generator.py
import pandas as pd
import os

def generate_submission(final_matches: pd.DataFrame, candidates_df: pd.DataFrame, s1_ids: list, output_dir: str):
    """
    Takes the final matches and the blocking candidates and formats them 
    according to the challenge submission requirements.
    Preserves all singletons by left-joining against the complete universe of S1 IDs.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Base dataframe containing all query IDs
    base_df = pd.DataFrame({'source1_entity_id': s1_ids})
    
    # 1. format matching_results.tsv
    matching = final_matches.groupby('s1_entity_id')['S2_entity_id'].apply(lambda x: ','.join(x)).reset_index()
    matching.columns = ['source1_entity_id', 'matched_entity_ids']
    
    final_matching = base_df.merge(matching, on='source1_entity_id', how='left')
    final_matching['matched_entity_ids'] = final_matching['matched_entity_ids'].fillna("")
    
    match_path = os.path.join(output_dir, "matching_results.tsv")
    final_matching.to_csv(match_path, sep='\t', index=False)
    print(f"Final matches saved to {match_path}")
    
    # 2. format candidate_pairs.tsv
    candidates = candidates_df.groupby('s1_entity_id')['S2_entity_id'].apply(lambda x: ','.join(x)).reset_index()
    candidates.columns = ['source1_entity_id', 'candidate_entity_ids']
    
    final_candidates = base_df.merge(candidates, on='source1_entity_id', how='left')
    final_candidates['candidate_entity_ids'] = final_candidates['candidate_entity_ids'].fillna("")
    
    cand_path = os.path.join(output_dir, "candidate_pairs.tsv")
    final_candidates.to_csv(cand_path, sep='\t', index=False)
    print(f"Candidate pairs saved to {cand_path}")
    
    return final_matching, final_candidates



In [ ]:
%%writefile src/pipeline.py
import pandas as pd
import numpy as np
import os
import gc
from src.data_loader import load_and_preprocess
from src.blocking.pipeline import BlockingPipeline
from src.features.feature_pipeline import build_features
from src.matching.lgbm_matcher import LGBMMatcher
from src.matching.singleton_detector import apply_singleton_threshold
from src.postprocessing.submission_generator import generate_submission

def run_end_to_end_pipeline(data_dir: str, output_dir: str, sample_size: int = None):
    """
    Runs the entire Amazon ML Challenge pipeline properly split into Train and Test.
    """
    print("=== AMAZON ML CHALLENGE ER PIPELINE ===")
    os.makedirs(output_dir, exist_ok=True)
    
    # =========================================================================
    # PHASE A: TRAINING
    # =========================================================================
    print("\n--- PHASE A: TRAINING ---")
    print("[1] Loading Training Data (Subsampled for RAM constraints)...")
    
    # For training, we must sample to fit in memory
    train_sample_size = sample_size if sample_size else 50000 
    df_s1_train = load_and_preprocess(f"{data_dir}/train/train_source1.tsv").head(train_sample_size)
    
    # Load full S2 and S3 for blocking
    df_s2_train = load_and_preprocess(f"{data_dir}/train/train_source2.tsv")
    df_s3_train = load_and_preprocess(f"{data_dir}/train/train_source3.tsv")
    df_gt = pd.read_csv(f"{data_dir}/train/train_ground_truth.tsv", sep='\t', na_filter=False)
    
    print("\n[2] Blocking Training Candidates...")
    blocker = BlockingPipeline(top_k=20, ngram_range=(3, 3))
    
    cand_s2_train = blocker.run(df_s1_train, df_s2_train, s2_prefix='S2')
    cand_s3_train = blocker.run(df_s1_train, df_s3_train, s2_prefix='S3')
    
    # Combine S2 and S3 candidates
    cand_s3_train.rename(columns={'S3_entity_id': 'S2_entity_id'}, inplace=True)
    candidates_train = pd.concat([cand_s2_train, cand_s3_train], ignore_index=True)
    
    print("\n[3] Generating Training Labels...")
    true_pairs = set()
    s1_ids = set(df_s1_train['entity_id'])
    
    df_gt_sample = df_gt[df_gt['source1_entity_id'].isin(s1_ids)]
    for _, row in df_gt_sample.iterrows():
        s1_id = row['source1_entity_id']
        matches_str = row['matched_entity_ids']
        if matches_str:
            for match in str(matches_str).split(','):
                true_pairs.add((s1_id, match))
                
    candidates_train['is_match'] = candidates_train.apply(
        lambda r: 1 if (r['s1_entity_id'], r['S2_entity_id']) in true_pairs else 0, axis=1
    )
    
    print("\n[4] Extracting Train Features...")
    df_s2_s3_combined = pd.concat([df_s2_train, df_s3_train], ignore_index=True)
    X_train = build_features(candidates_train, df_s1_train, df_s2_s3_combined, s2_prefix='S2')
    y_train = candidates_train['is_match'].values
    
    print("\n[5] Training LightGBM Matcher...")
    matcher = LGBMMatcher()
    matcher.fit(X_train, y_train)
    
    # Clear RAM
    del df_s1_train, df_s2_train, df_s3_train, candidates_train, X_train, y_train, df_s2_s3_combined
    gc.collect()
    
    # =========================================================================
    # PHASE B: INFERENCE (TEST SET)
    # =========================================================================
    print("\n--- PHASE B: INFERENCE (TEST) ---")
    print("[1] Loading Test Data...")
    df_s1_test = load_and_preprocess(f"{data_dir}/test/test_source1.tsv")
    df_s2_test = load_and_preprocess(f"{data_dir}/test/test_source2.tsv")
    df_s3_test = load_and_preprocess(f"{data_dir}/test/test_source3.tsv")
    
    if sample_size:
        df_s1_test = df_s1_test.head(sample_size).copy()
        
    s1_universe = df_s1_test['entity_id'].tolist()
    
    print("\n[2] Blocking Test Candidates...")
    cand_s2_test = blocker.run(df_s1_test, df_s2_test, s2_prefix='S2')
    cand_s3_test = blocker.run(df_s1_test, df_s3_test, s2_prefix='S3')
    
    cand_s3_test.rename(columns={'S3_entity_id': 'S2_entity_id'}, inplace=True)
    candidates_test = pd.concat([cand_s2_test, cand_s3_test], ignore_index=True)
    
    print("\n[3] Extracting Test Features...")
    df_s2_s3_test = pd.concat([df_s2_test, df_s3_test], ignore_index=True)
    X_test = build_features(candidates_test, df_s1_test, df_s2_s3_test, s2_prefix='S2')
    
    print("\n[4] Predicting & Applying Thresholds...")
    candidates_test['match_prob'] = matcher.predict_proba(X_test)
    final_matches = apply_singleton_threshold(candidates_test, singleton_threshold=0.6, match_threshold=0.4)
    
    print("\n[5] Generating Submission Files...")
    matching, candidates = generate_submission(final_matches, candidates_test, s1_universe, output_dir)
    
    print("Pipeline Execution Complete!")
    return matching



In [ ]:
# KAGGLE EXECUTION SCRIPT
import pandas as pd
from src.pipeline import run_end_to_end_pipeline

# NOTE: Change this to your Kaggle Dataset path!
# Example: DATA_DIR = "/kaggle/input/amazon-ml-challenge-2026/dataset"
DATA_DIR = "6ab10eb3b23ba_student_resource/student_resource/dataset"
OUTPUT_DIR = "submission"

# We run on a 10,000 record sample for testing. 
# Remove sample_size=10000 for the full run!
final_matches = run_end_to_end_pipeline(data_dir=DATA_DIR, output_dir=OUTPUT_DIR, sample_size=10000)

print(final_matches.head())
